In [2]:
import pandas as pd
import os
import json
import math

# --- Settings ---
csv_file = 'df_clean_integrated.csv'
output_files_count = 10  # Number of JSONL files
target_file_size_mb = 90  # Approximate size per file
output_prefix = csv_file.replace('.csv', '') + '_sample'

# --- Step 1: Read CSV ---
df = pd.read_csv(csv_file)
total_rows = len(df)
print(f"CSV file '{csv_file}' loaded with {total_rows} rows.")

# --- Step 2: Estimate average row size ---
sample_size = min(1000, total_rows)
sample_json_lines = [json.dumps(row) for row in df.head(sample_size).to_dict(orient='records')]
avg_row_size_bytes = sum(len(line.encode('utf-8')) for line in sample_json_lines) / sample_size
print(f"Estimated average row size: {avg_row_size_bytes:.2f} bytes")

# --- Step 3: Calculate total sample rows needed ---
rows_per_file = int((target_file_size_mb * 1024 * 1024) / avg_row_size_bytes)
total_sample_rows = rows_per_file * output_files_count
print(f"Will sample {total_sample_rows} rows total (~{target_file_size_mb} MB per file)")

# --- Step 4: Sample rows ---
df_sample = df.sample(n=min(total_sample_rows, total_rows), random_state=42)
total_rows_sample = len(df_sample)
print(f"Sampled {total_rows_sample} rows for splitting")

# --- Step 5: Split and write JSONL files ---
rows_per_file = math.ceil(total_rows_sample / output_files_count)

for i, start in enumerate(range(0, total_rows_sample, rows_per_file), 1):
    chunk = df_sample.iloc[start:start + rows_per_file]
    jsonl_file = f"{output_prefix}_part{i}.jsonl"
    with open(jsonl_file, 'w', encoding='utf-8') as f:
        for record in chunk.to_dict(orient='records'):
            f.write(json.dumps(record) + '\n')
    size_mb = os.path.getsize(jsonl_file) / (1024 * 1024)
    print(f"Written '{jsonl_file}' ({size_mb:.2f} MB)")

print("Sample JSONL files created successfully.")

CSV file 'df_clean_integrated.csv' loaded with 4849276 rows.
Estimated average row size: 1247.26 bytes
Will sample 756630 rows total (~90 MB per file)
Sampled 756630 rows for splitting
Written 'df_clean_integrated_sample_part1.jsonl' (89.56 MB)
Written 'df_clean_integrated_sample_part2.jsonl' (89.56 MB)
Written 'df_clean_integrated_sample_part3.jsonl' (89.56 MB)
Written 'df_clean_integrated_sample_part4.jsonl' (89.56 MB)
Written 'df_clean_integrated_sample_part5.jsonl' (89.57 MB)
Written 'df_clean_integrated_sample_part6.jsonl' (89.57 MB)
Written 'df_clean_integrated_sample_part7.jsonl' (89.56 MB)
Written 'df_clean_integrated_sample_part8.jsonl' (89.55 MB)
Written 'df_clean_integrated_sample_part9.jsonl' (89.55 MB)
Written 'df_clean_integrated_sample_part10.jsonl' (89.56 MB)
Sample JSONL files created successfully.


In [1]:
import pandas as pd
import os
import json

# --- Settings ---
csv_file = 'df_clean_integrated.csv'
max_file_size_mb = 50  # Maximum JSONL file size
output_prefix = csv_file.replace('.csv', '')  # Base name for output files

# --- Step 1: Read CSV ---
df = pd.read_csv(csv_file)
print(f"CSV file '{csv_file}' loaded with {len(df)} rows.")

# --- Step 2: Estimate chunk size ---
# Convert a sample of rows to JSON strings to estimate size
sample_size = min(1000, len(df))
sample_json_lines = [json.dumps(row) for row in df.head(sample_size).to_dict(orient='records')]
avg_row_size_bytes = sum(len(line.encode('utf-8')) for line in sample_json_lines) / sample_size
rows_per_file = int((max_file_size_mb * 1024 * 1024) / avg_row_size_bytes)
print(f"Estimated {rows_per_file} rows per JSONL file (to stay under {max_file_size_mb} MB).")

# --- Step 3: Split and write JSONL files ---
num_files = (len(df) + rows_per_file - 1) // rows_per_file
print(f"Will generate {num_files} JSONL file(s).")

for i, start in enumerate(range(0, len(df), rows_per_file), 1):
    chunk = df.iloc[start:start+rows_per_file]
    jsonl_file = f"{output_prefix}_part{i}.jsonl"
    with open(jsonl_file, 'w', encoding='utf-8') as f:
        for record in chunk.to_dict(orient='records'):
            f.write(json.dumps(record) + '\n')
    size_mb = os.path.getsize(jsonl_file) / (1024*1024)
    print(f"Written '{jsonl_file}' ({size_mb:.2f} MB)")

print("All JSONL files created successfully.")


CSV file 'df_clean_integrated.csv' loaded with 4849276 rows.
Estimated 42035 rows per JSONL file (to stay under 50 MB).
Will generate 116 JSONL file(s).
Written 'df_clean_integrated_part1.jsonl' (50.12 MB)
Written 'df_clean_integrated_part2.jsonl' (50.01 MB)
Written 'df_clean_integrated_part3.jsonl' (49.80 MB)


KeyboardInterrupt: 